# AI-Based Student Performance Prediction and Risk Analysis System
## Complete Machine Learning Workflow Notebook for MCA Academic Demonstration

**Project Objectives:**
1. Data Preprocessing (Missing value imputation, duplicate removal, IQR outlier capping)
2. Categorical Encoding & Standard Feature Scaling
3. Regression Modeling (Predicting Final Exam Marks)
4. Classification Modeling (Predicting Pass/Fail Outcome across 5 algorithms)
5. Unsupervised K-Means Clustering (Optimal K via Elbow Method)
6. Student Academic Risk Categorization Engine
7. Model Serialization using Joblib for Web Deployment

### Step 1: Import Required Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# Set plot styles
plt.style.use('ggplot')
sns.set_theme(style='whitegrid')
print('All libraries imported successfully!')

### Step 2 & 3: Load and Inspect Dataset

In [2]:
csv_path = os.path.join('..', 'data', 'student_data.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.join('data', 'student_data.csv')

df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [3]:
df.info()
df.describe()

### Step 4, 5, 6 & 7: Data Preprocessing (Missing Values, Duplicates, Outliers)

In [4]:
# 1. Missing Values Detection
print("Missing Values Count Before Imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Numeric Imputation using Median
num_cols = ['age', 'attendance', 'study_hours', 'assignment_score', 'internal_marks', 'previous_semester_marks', 'extracurricular_hours', 'sleep_hours', 'backlogs']
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Categorical Imputation using Mode
df['gender'] = df['gender'].fillna('Male')
df['internet_access'] = df['internet_access'].fillna('Yes')
df['result'] = df['result'].fillna('PASS')

# 2. Duplicate Removal
dup_count = df.duplicated().sum()
print(f"Duplicates Detected & Removed: {dup_count}")
df = df.drop_duplicates().reset_index(drop=True)

# 3. Outlier Handling via IQR Capping
for col in ['study_hours', 'sleep_hours', 'extracurricular_hours']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower, upper)

print("Data cleaning completed successfully!")

### Step 8 & 9: Encoding & Feature Scaling

In [5]:
# Categorical Mapping
df['gender_encoded'] = df['gender'].map({'Male': 1, 'Female': 0})
df['internet_access_encoded'] = df['internet_access'].map({'Yes': 1, 'No': 0})
df['result_encoded'] = df['result'].map({'PASS': 1, 'FAIL': 0})

feature_cols = ['age', 'gender_encoded', 'attendance', 'study_hours', 'assignment_score', 'internal_marks', 'previous_semester_marks', 'extracurricular_hours', 'internet_access_encoded', 'sleep_hours', 'backlogs']

X = df[feature_cols]
y_reg = df['final_marks']
y_clf = df['result_encoded']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
print("Feature scaling completed!")

### Step 10 & 11: Exploratory Data Analysis & Train/Test Split

In [6]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[feature_cols + ['final_marks']].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.show()

# Split dataset
X_train, X_test, y_train_reg, y_test_reg = train_test_split(X_scaled_df, y_reg, test_size=0.2, random_state=42)
_, _, y_train_clf, y_test_clf = train_test_split(X_scaled_df, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

### Step 12: Regression Models (Predicting Final Marks)

In [7]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train_reg)
pred_lr = lr_model.predict(X_test)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train_reg)
pred_rf_reg = rf_reg.predict(X_test)

print("=== Linear Regression Metrics ===")
print(f"MSE: {mean_squared_error(y_test_reg, pred_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, pred_lr)):.4f}")
print(f"MAE: {mean_absolute_error(y_test_reg, pred_lr):.4f}")
print(f"R2 Score: {r2_score(y_test_reg, pred_lr):.4f}")

print("\n=== Random Forest Regressor Metrics ===")
print(f"MSE: {mean_squared_error(y_test_reg, pred_rf_reg):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, pred_rf_reg)):.4f}")
print(f"MAE: {mean_absolute_error(y_test_reg, pred_rf_reg):.4f}")
print(f"R2 Score: {r2_score(y_test_reg, pred_rf_reg):.4f}")

### Step 13 & 14: Classification Models & Comparison

In [8]:
clf_models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine': SVC(probability=True, random_state=42),
    'Naive Bayes': GaussianNB()
}

clf_results = []
for name, model in clf_models.items():
    model.fit(X_train, y_train_clf)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test_clf, preds)
    prec = precision_score(y_test_clf, preds, zero_division=0)
    rec = recall_score(y_test_clf, preds, zero_division=0)
    f1 = f1_score(y_test_clf, preds, zero_division=0)
    clf_results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4)
    })

comparison_df = pd.DataFrame(clf_results)
comparison_df

### Step 15 & 16: K-Means Clustering & Visualizations

In [9]:
cluster_features = ['attendance', 'study_hours', 'assignment_score', 'internal_marks', 'previous_semester_marks', 'final_marks']
X_cluster = StandardScaler().fit_transform(df[cluster_features])

wcss = []
for k in range(1, 11):
    kmeans_t = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_t.fit(X_cluster)
    wcss.append(kmeans_t.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), wcss, marker='o', color='blue')
plt.title('Elbow Method')
plt.xlabel('K Clusters')
plt.ylabel('WCSS')
plt.show()

# Fit K=3
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_id'] = kmeans_final.fit_predict(X_cluster)
print("Cluster counts:")
print(df['cluster_id'].value_counts())

### Step 17: Model Saving with Joblib

In [10]:
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))
joblib.dump(lr_model, os.path.join(models_dir, 'regression_model.pkl'))
joblib.dump(clf_models['Support Vector Machine'], os.path.join(models_dir, 'classification_model.pkl'))
print("All trained models exported to models/ folder!")